This notebook tests loading some ERA5 data from the ARCO-ERA5 



In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""

Downloads ERA5 reanalysis data for Western Europe from the ARCO-ERA5 public
Google Cloud Storage bucket and saves it as a local Zarr store for efficient
downstream analysis.

Source  : gs://gcp-public-data-arco-era5/co/single-level-reanalysis.zarr
Region  : France, Belgium, Netherlands, Germany (41–55°N, 3–16°E)
Period  : 1980-01-01 to 2023-12-31
Variables:
    - t2m   : 2m air temperature (K)
    - d2m   : 2m dewpoint temperature (K)
    - swvl1 : Volumetric soil water layer 1, 0–7 cm (m³/m³)
    - swvl2 : Volumetric soil water layer 2, 7–28 cm (m³/m³)

Output  : ./Data/ERA5_zarr/ERA5_FRBENLDE.zarr

NOTE: Run this script once before any analysis scripts. Requires a stable
internet connection. Data volume is substantial — expect 30–90 min depending
on connection speed.

@author: afer
"""


'\n\nDownloads ERA5 reanalysis data for Western Europe from the ARCO-ERA5 public\nGoogle Cloud Storage bucket and saves it as a local Zarr store for efficient\ndownstream analysis.\n\nSource  : gs://gcp-public-data-arco-era5/co/single-level-reanalysis.zarr\nRegion  : France, Belgium, Netherlands, Germany (41–55°N, 3–16°E)\nPeriod  : 1980-01-01 to 2023-12-31\nVariables:\n    - t2m   : 2m air temperature (K)\n    - d2m   : 2m dewpoint temperature (K)\n    - swvl1 : Volumetric soil water layer 1, 0–7 cm (m³/m³)\n    - swvl2 : Volumetric soil water layer 2, 7–28 cm (m³/m³)\n\nOutput  : ./Data/ERA5_zarr/ERA5_FRBENLDE.zarr\n\nNOTE: Run this script once before any analysis scripts. Requires a stable\ninternet connection. Data volume is substantial — expect 30–90 min depending\non connection speed.\n\n@author: afer\n'

### imports

In [2]:

import numpy as np
import xarray as xr
import zarr
import os
import fsspec
import gcsfs
from pathlib import Path


## Configuration and parameters for file stream 


In [ ]:

ARCO_URL   = 'gs://gcp-public-data-arco-era5/co/single-level-reanalysis.zarr'
ZARR_PATH  = Path('./../../Data/ERA5_FRBENLDE_zarr/ERA5_FRBENLDE.zarr')
# VARIABLES  = ['t2m', 'd2m', 'swvl1', 'swvl2']
# VARIABLES  = ['swvl1', 'swvl2']
VARIABLES  = ['t2m', 'd2m']
TIME_START = '1980-01-01'
TIME_END   = '2023-12-31'

# Bounding box for FR, BE, NL, DE
# ARCO-ERA5 uses 0–360° longitude convention; Western Europe needs no conversion
LAT_MIN, LAT_MAX =  41,   55
LON_MIN, LON_MAX =   3,   16


print("Connecting to ARCO-ERA5 bucket...")
reanalysis = xr.open_zarr(
    ARCO_URL,
    chunks={'time': 48},
    consolidated=True,
)


Connecting to ARCO-ERA5 bucket...


/var/folders/p4/7cgg9ywn2hz45fzv3s976d_00000gn/T/ipykernel_96516/3236389985.py:15: FutureWarning: In a future version, xarray will not decode the variable 'step' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  reanalysis = xr.open_zarr(


### temporal subset

In [4]:
print(f"Subsetting time: {TIME_START} to {TIME_END}...")
subset = reanalysis[VARIABLES].sel(
    time=slice(TIME_START, TIME_END)
)


Subsetting time: 1980-01-01 to 2023-12-31...



### Spatial subset on flat 'values' dimension 

ARCO-ERA5 stores data on an unstructured grid: latitude and longitude are
1D auxiliary coordinates indexed by a flat 'values' dimension. We identify
the indices that fall within the bounding box and select them directly.


In [5]:

print("Loading flat lat/lon coordinate arrays...")
lat = reanalysis.latitude.compute().values   # shape: (542080,)
lon = reanalysis.longitude.compute().values  # shape: (542080,)

spatial_mask = (
    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
    (lon >= LON_MIN) & (lon <= LON_MAX)
)
values_idx = np.where(spatial_mask)[0]
print(f"Grid points inside bounding box: {len(values_idx)}")

subset_spatial = subset.isel(values=values_idx)


Loading flat lat/lon coordinate arrays...
Grid points inside bounding box: 1627


In [6]:
print(ZARR_PATH)
print(os.getcwd())

../../Data/ERA5_FRBENLDE_zarr/ERA5_FRBENLDE.zarr
/Users/afer/CE/ag/Code/notebooks


In [7]:
# Download raw data preserving the native 'values' dimension
# No set_index, no unstack — just spatial and temporal subsetting

from dask.diagnostics import ProgressBar

subset_spatial = reanalysis[VARIABLES].sel(
    time=slice(TIME_START, TIME_END)
).isel(values=values_idx)

# Assign lat/lon as auxiliary coordinates but DO NOT unstack
subset_spatial = subset_spatial.assign_coords(
    latitude=('values',  lat[values_idx]),
    longitude=('values', lon[values_idx])
)

# Save per variable
for var in VARIABLES:
    var_zarr_path = ZARR_PATH / f"ERA5_FRBENLDE_{var}_raw.zarr"
    print(f"Writing {var}...")
    with ProgressBar():
        (
            subset_spatial[[var]]
            .chunk({'time': 24 * 365, 'values': -1})
            .to_zarr(var_zarr_path, mode='w', compute=True)
        )

Writing swvl1...
[########################################] | 100% Completed | 3hr 22m
Writing swvl2...
[########################################] | 100% Completed | 2hr 34m


In [8]:
ZARR_PATH.parent.mkdir(parents=True, exist_ok=True)

for var in VARIABLES:
    var_zarr_path = ZARR_PATH.parent / f"ERA5_FRBENLDE_{var}.zarr"
    if var_zarr_path.exists():
        # Check how many bytes have been written so far
        size_gb = sum(
            f.stat().st_size for f in var_zarr_path.rglob('*') if f.is_file()
        ) / 1e9
        print(f"{var}: {size_gb:.2f} GB written so far")
    else:
        print(f"{var}: not started yet")


swvl1: 4.64 GB written so far
swvl2: 4.27 GB written so far


In [ ]:
for var in VARIABLES:
    var_zarr_path = ZARR_PATH.parent / f"ERA5_FRBENLDE_{var}.zarr"
    ds = xr.open_zarr(str(var_zarr_path))
    print(f"\n{var}:")
    print(f"  dims   : {dict(ds.dims)}")
    print(f"  time   : {str(ds.valid_time.values[0])[:10]} to {str(ds.valid_time.values[-1])[:10]}")
    print(f"  lat    : {float(ds.latitude.min()):.2f} to {float(ds.latitude.max()):.2f}")
    print(f"  lon    : {float(ds.longitude.min()):.2f} to {float(ds.longitude.max()):.2f}")
